# MDTF Variable Translation: Developer Guide

> Disclaimer: This notebook was created with GitHub Copilot assistance and is still being vetted.

This notebook introduces the framework's variable-name translation machinery. It is for code exploration and small examples; it does not launch a full MDTF run or modify the fieldlist files. Run it from a kernel with the MDTF repository dependencies available.

For a reusable template that you can adapt for future POD examples, see [variable_translation_template.ipynb](variable_translation_template.ipynb).

## The Main Idea

POD settings describe requested variables. Fieldlist tables describe names and metadata used by data conventions. The translator loads those tables and routes lookups to the fieldlist for a convention. In a framework run, `translate_data` decides whether the data convention is used or whether the special `no_translation` fieldlist is selected.

The main runtime path is:

1. [`mdtf_framework.py`](../../mdtf_framework.py#L159) loads translator tables and [`mdtf_framework.py`](../../mdtf_framework.py#L185) starts POD setup.
2. [`PodObject.setup_pod`](../../src/pod_setup.py#L313) checks `runtime_config.translate_data` and chooses the target convention.
3. [`DataSourceBase.translate_varlist`](../../src/data_sources.py#L100) forwards the variable to its varlist.
4. [`Varlist.setup_var`](../../src/varlist_util.py#L547) selects a fieldlist and calls that fieldlist's `translate()` method.

The `VariableTranslator.translate()` wrapper is a separate convenience method. The gated framework path above calls the selected fieldlist directly. See [`translation.py`](../../src/translation.py#L634) for the wrapper implementation.

## Find The Repository Root

The source package is named `src`. Find the checkout from the notebook's current directory so imports work whether Jupyter started in the repository root or a subdirectory.

This notebook imports MDTF internals directly from the repository's `src` package. Until a lightweight, pip-installable MDTF API is exposed, it must locate the checkout and add its root to `sys.path` so those imports work when Jupyter is started from a subdirectory. This is a temporary development workaround; consumers of the future installed API should not need to manage repository paths.

In [ ]:
import sys
from pathlib import Path

# Find the repository root directory for the MDTF checkout
candidate_dirs = (Path.cwd(), *Path.cwd().parents)
repo_root = next(
    (path for path in candidate_dirs if (path / "src" / "translation.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Start Jupyter in the MDTF checkout or one of its subdirectories.")

# Adds the repository root to the system path so that the translation module can be imported
sys.path.insert(0, str(repo_root))
print(repo_root)

## Load the Fieldlists

`VariableTranslator` is a singleton in this codebase. The framework initializes it once with `repo_root`, then reads `data/fieldlist_*.jsonc`. The field lists contain variable attributes for each convention (e.g., CMIP, GFDL, CESM). For an isolated notebook kernel, use the same initialization sequence below.

In [ ]:
from src.translation import VariableTranslator

translator = VariableTranslator(str(repo_root))
translator.read_conventions(str(repo_root))

print("Loaded conventions:", sorted(translator.conventions.keys()))

## Verify The GFDL-to-CMIP Name Mapping

First ask the translator for the CF standard name of GFDL `t_ref`. Then look up that standard name in CMIP using the same realm and modifier. This isolates the fieldlist mapping before translating a full `VarlistEntry`.

`lut` is short for lookup table. In this code, it’s a dictionary on the convention’s fieldlist: variable names are keys, and their metadata are the values.



In [ ]:
gfdl_t_ref = translator.get_convention("GFDL").lut["t_ref"]
cmip_tas = translator.get_convention("CMIP").lut["tas"]

print("GFDL t_ref:", gfdl_t_ref["standard_name"], gfdl_t_ref["modifier"], cmip_tas["realm"])
print("CMIP tas:", cmip_tas["standard_name"], cmip_tas["modifier"], cmip_tas["realm"])
assert gfdl_t_ref["standard_name"] == cmip_tas["standard_name"]
assert gfdl_t_ref["realm"] == cmip_tas["realm"]
assert gfdl_t_ref["modifier"] == cmip_tas["modifier"]

## Translate From CMIP Varlist Entry To GFDL

At runtime, `Varlist.setup_var()` gets the data convention's `Fieldlist` and calls its `translate(var, from_convention)` method. This example follows that route with a CMIP `VarlistEntry` and the GFDL target fieldlist. The matching entries share `air_temperature`, realm `atmos`, units `K`, three dimensions, and modifier `atmos_height`, so the translated name should be `t_ref`.

For standalone use, `VariableTranslator.translate(var, to_convention, from_convention)` is a wrapper that selects the fieldlist and calls the same method.

In [ ]:
from types import SimpleNamespace
from src.varlist_util import Varlist

# The example_settings are akin to the settings.json in the v4 framework.
example_settings = {
    "data": {"frequency": "day", "realm": "atmos"},
    "dimensions": {
        "lat": {
            "axis": "Y",
            "standard_name": "latitude",
            "units": "degrees_north",
        },
        "lon": {
            "axis": "X",
            "standard_name": "longitude",
            "units": "degrees_east",
        },
        "time": {"axis": "T", "standard_name": "time", "units": "days"},
    },
    "varlist": {
        "tas": {
            "standard_name": "air_temperature",
            "long_name": "Near-Surface Air Temperature",
            "units": "K",
            "modifier": "atmos_height",
            "dimensions": ["time", "lat", "lon"],
        }
    },
}
pod = SimpleNamespace(
    pod_data=example_settings["data"],
    pod_dims=example_settings["dimensions"],
    pod_vars=example_settings["varlist"],
    pod_settings={"convention": "CMIP"},
    _log_name="translation_notebook",
)
cmip_tas_var = Varlist.from_struct(pod).vars[0]
gfdl_fieldlist = translator.get_convention("GFDL")
translated_var = gfdl_fieldlist.translate(cmip_tas_var, "CMIP")
translated_via_wrapper = translator.translate(
    cmip_tas_var, to_convention="GFDL", from_convention="CMIP"
)

print("Requested:", cmip_tas_var.name, cmip_tas_var.standard_name)
print("Framework path:", translated_var.name, translated_var.standard_name)
assert translated_var.name == "t_ref"
assert translated_via_wrapper.name == translated_var.name

## How The Runtime Gate Works

In [`src/pod_setup.py`](../../src/pod_setup.py#L313), each case starts with its configured data convention. If `translate_data` is false, setup substitutes `no_translation`; otherwise the configured data convention remains the target. Setup passes the POD convention and chosen target convention into `translate_varlist()`.

For the example translation here, those values are `pod_convention="CMIP"`, `data_convention="GFDL"`, and `translate_data=True`. The framework therefore selects the GFDL fieldlist and passes the CMIP convention as the POD source.

In [`src/varlist_util.py`](../../src/varlist_util.py#L547), `VariableTranslator().get_convention(to_convention)` returns the data fieldlist, or a `NoTranslationFieldlist` for the sentinel. The code then calls `translate.translate(v, from_convention)`. So the switch affects which fieldlist implementation runs; it does not skip the varlist setup call.

Keep the two APIs distinct when debugging:

- `VariableTranslator.translate(var, to_convention, from_convention)` is a wrapper that chooses a fieldlist and calls its `translate()`.
- The `translate_data` runtime path calls `Fieldlist.translate(var, from_convention)` on the fieldlist already selected in `Varlist.setup_var()`.
- Preprocessor request-editing has a direct `VariableTranslator.translate()` call at [`src/preprocessor.py`](../../src/preprocessor.py#L206); that is another use case.

## Where To Go Next

- Read [`src/translation.py`](../../src/translation.py#L517) for convention loading, aliases, and fieldlist routing.
- Read [`src/translation.py`](../../src/translation.py#L363) for the convention-specific `Fieldlist.translate()` implementation.
- Inspect [`data/fieldlist_CMIP.jsonc`](../../data/fieldlist_CMIP.jsonc) and the other `fieldlist_*.jsonc` files to understand translation metadata.
- Run the focused tests with `python -m unittest src.tests.test_translation.TestVariableTranslatorFiles.test_variabletranslator_real_data` from the repository root.

## Exercise the Framework Preprocessor for the Multicase Example

The next example follows a runtime request: the POD requests CMIP `tas`, while the prepared GFDL files contain `t_ref`. It opens both prepared case datasets lazily, applies the framework's `ConvertUnitsFunction` and `RenameVariablesFunction`, then uses `MDTFPreprocessorBase.rename_dataset_vars()` to map the dependent variable back to the POD name. It follows the multirun demo's calculation and plot pattern: time mean, shuffled-field global-mean anomaly, zonal mean, and a latitude profile for each case.

This variable pair demonstrates the overall preprocessing path, but not a nontrivial unit or coordinate conversion: both conventions specify `K`, and their coordinate names are `time`, `lat`, and `lon`. Thus `ConvertUnitsFunction` makes no numerical change and `RenameVariablesFunction` has no coordinate labels to change. The data-variable translation from `t_ref` to `tas` is meaningful and is performed by the later `rename_dataset_vars()` step. The shuffle is only a visual demonstration and should not be interpreted scientifically. The prepared files are several gigabytes in total, so they are opened lazily and only the reduced latitude profiles are loaded into memory.

In [ ]:
from types import SimpleNamespace

from src.preprocessor import (
    ConvertUnitsFunction,
    MDTFPreprocessorBase,
    RenameVariablesFunction,
)
from src.varlist_util import Varlist

pod_tas_field = translator.get_convention("CMIP").to_CF("tas")
runtime_settings = {
    "data": {"frequency": "day", "realm": pod_tas_field.get("realm") or "atmos"},
    "dimensions": {
        "lat": {
            "axis": "Y",
            "standard_name": "latitude",
            "units": "degrees_north",
        },
        "lon": {
            "axis": "X",
            "standard_name": "longitude",
            "units": "degrees_east",
        },
        "time": {"axis": "T", "standard_name": "time", "units": "days"},
    },
    "varlist": {
        "tas": {
            "standard_name": pod_tas_field["standard_name"],
            "long_name": pod_tas_field.get("long_name", ""),
            "units": pod_tas_field["units"],
            "modifier": pod_tas_field.get("modifier", ""),
            "dimensions": ["time", "lat", "lon"],
        }
    },
}
runtime_pod = SimpleNamespace(
    name="framework_preprocessing_example",
    pod_data=runtime_settings["data"],
    pod_dims=runtime_settings["dimensions"],
    pod_vars=runtime_settings["varlist"],
    pod_settings={"convention": "CMIP"},
    _log_name="framework_preprocessing_example",
)
pod_tas = Varlist.from_struct(runtime_pod).vars[0]
pod_tas.translation = translator.get_convention("GFDL").translate(pod_tas, "CMIP")
source_name = pod_tas.translation.name
assert source_name == "t_ref"

print(f"POD request: {pod_tas.name}; GFDL data variable: {source_name}")

### Load the prepared GFDL case datasets

The case YAML contains paths from another checkout. Resolve each file by its basename inside this repository's `prepared_gfdl_data` directory, and open it lazily so the large NetCDF files are not loaded into memory at once.

In [ ]:
from pathlib import Path
import yaml
import xarray as xr

example_dir = repo_root / "diagnostics" / "example_multicase"
case_info_path = example_dir / "case_info_gfdl_demo.yml"
with case_info_path.open() as stream:
    case_info = yaml.safe_load(stream)

source_datasets = {}
for case_name, case_config in case_info["CASE_LIST"].items():
    data_path = example_dir / "prepared_gfdl_data" / Path(case_config["T_REF_FILE"]).name
    if not data_path.is_file():
        raise FileNotFoundError(f"Prepared data file not found: {data_path}")

    dataset = xr.open_dataset(
        data_path,
        chunks={case_config["time_coord"]: 1},
        decode_times=True,
        use_cftime=True,
    )
    source_datasets[case_name] = dataset
    print(f"{case_name}: {data_path.name}, sizes={dict(dataset.sizes)}")

### Run framework preprocessing

Convert units and reconcile translated coordinate names, then apply the framework's final dependent-variable rename.

In [ ]:
coords_renamed = {}
case_views = {}
for case_name, dataset in source_datasets.items():
    converted = ConvertUnitsFunction().execute(pod_tas, dataset)
    coords_renamed[case_name] = RenameVariablesFunction().execute(pod_tas, converted)
    case_views[case_name] = SimpleNamespace(
        varlist=SimpleNamespace(iter_vars=lambda: [pod_tas])
    )

pod_named_datasets = MDTFPreprocessorBase.rename_dataset_vars(
    None,
    coords_renamed,
    case_views,
)
for case_name, dataset in pod_named_datasets.items():
    assert pod_tas.name in dataset.data_vars
    print(f"{case_name}: {source_name} -> {pod_tas.name}; sizes={dict(dataset.sizes)}")

### Calculate shuffled zonal-mean anomalies

Like the multirun demo, randomly shuffle each case's time-mean field before calculating anomalies and zonal means. This makes the example plot visually interesting, but destroys the geographic pattern; do not interpret these shuffled profiles as scientific results. This is for example purposes only.

In [ ]:
import numpy as np

np.random.seed(0)
tas_arrays = {}
for case_name, dataset in pod_named_datasets.items():
    case_config = case_info["CASE_LIST"][case_name]
    tas = dataset[pod_tas.name].mean(dim=case_config["time_coord"])

    # Match the multirun demo: shuffle the time-mean field before anomalies.
    tas.load()
    shuffled_values = tas.to_masked_array().flatten()
    np.random.shuffle(shuffled_values)
    tas.values = shuffled_values.reshape(tas.shape)

    tas_anomaly = tas - tas.mean()
    tas_zonal_mean = tas_anomaly.mean(dim=case_config["lon_coord"])
    tas_arrays[case_name] = tas_zonal_mean
    print(f"Calculated shuffled zonal-mean anomaly for {case_name}")

for dataset in source_datasets.values():
    dataset.close()

tas_arrays

### Plot zonal-mean anomalies

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

fig, ax = plt.subplots(figsize=(12, 4))
for case_name, latitude_profile in tas_arrays.items():
    latitude_profile.plot(ax=ax, label=case_name)

ax.set_title("Zonal Mean Surface Air Temperature Anomaly")
ax.set_xlabel("Latitude")
ax.set_ylabel(f"{pod_tas.name} anomaly ({pod_tas.units})")
ax.legend(title="GFDL case")
plt.tight_layout()
display(fig)

## Appendix: Validate a Unit Conversion

The main walkthrough focuses on variable translation. This optional check shows a numerical unit conversion alongside it: CMIP `tas` is translated to GFDL `t_ref`, then a tiny synthetic dataset is converted from kelvin to Celsius. The assertions check the translated name, expected values, and output units without opening prepared data.

In [ ]:
import numpy as np
import xarray as xr

conversion_settings = {
    "data": runtime_settings["data"],
    "dimensions": runtime_settings["dimensions"],
    "varlist": {
        "tas": {**runtime_settings["varlist"]["tas"], "units": "degC"}
    },
}
conversion_pod = SimpleNamespace(
    name="unit_conversion_example",
    pod_data=conversion_settings["data"],
    pod_dims=conversion_settings["dimensions"],
    pod_vars=conversion_settings["varlist"],
    pod_settings={"convention": "CMIP"},
    _log_name="unit_conversion_example",
)
conversion_var = Varlist.from_struct(conversion_pod).vars[0]
conversion_var.translation = translator.get_convention("GFDL").translate(
    conversion_var, "CMIP"
)
assert conversion_var.translation.name == "t_ref"

synthetic_gfdl = xr.Dataset(
    {
        "t_ref": (
            ("time", "lat", "lon"),
            [[[273.15, 293.15]]],
            {"standard_name": "air_temperature", "units": "K"},
        )
    },
    coords={
        "time": ("time", [0], {"standard_name": "time", "units": "days"}),
        "lat": ("lat", [0.0], {"standard_name": "latitude", "units": "degrees_north"}),
        "lon": ("lon", [0.0, 180.0], {"standard_name": "longitude", "units": "degrees_east"}),
    },
)
converted_synthetic = ConvertUnitsFunction().execute(
    conversion_var, synthetic_gfdl
)
expected_celsius = np.array([[[0.0, 20.0]]])
np.testing.assert_allclose(
    converted_synthetic["t_ref"].values, expected_celsius, atol=1e-6
)
assert converted_synthetic["t_ref"].attrs["units"] == str(
    conversion_var.units.units
)
print("Translated variable: t_ref -> tas")
print("Converted values (degC):", converted_synthetic["t_ref"].values.ravel())
print("Output units:", converted_synthetic["t_ref"].attrs["units"])